# **競賽說明**

本次的項目是要預測句子所帶有的情緒.<br>

本次項目為NLP–單一文本分類任務<br>

競賽網址:<br>
https://www.kaggle.com/competitions/sentiment-analysis-on-movie-reviews/overview

## **kernel架構:**

1. 讀取套件及載入資料
2. EDA 
3. Data Preprocessing(資料前處理) 
4. 模型建構 

問題思考和想法:<br>

- x

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sentiment-analysis-on-movie-reviews/sampleSubmission.csv
/kaggle/input/sentiment-analysis-on-movie-reviews/train.tsv.zip
/kaggle/input/sentiment-analysis-on-movie-reviews/test.tsv.zip


# **1. 讀取套件及載入資料**

In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

#skleran
#from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

#transformers 
from transformers import BertTokenizer
from transformers import TFBertForSequenceClassification
from transformers import TFBertModel

#tensorflow
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical,plot_model 
from keras.models import Sequential
from keras.layers import Dense,Activation,Dropout 

#other
#import missingno as msno
from IPython.display import clear_output

In [3]:
#tsv檔的間隔符與csv檔不同
#此處讀取檔案時需修改參數sep(間隔符)值
train=pd.read_csv('/kaggle/input/sentiment-analysis-on-movie-reviews/train.tsv.zip',sep='\t')
test=pd.read_csv('/kaggle/input/sentiment-analysis-on-movie-reviews/test.tsv.zip',sep='\t')

# **2. EDA**

## **2.1 資料概述**

In [4]:
print(train.info())
train.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156060 entries, 0 to 156059
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   PhraseId    156060 non-null  int64 
 1   SentenceId  156060 non-null  int64 
 2   Phrase      156060 non-null  object
 3   Sentiment   156060 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 4.8+ MB
None


,PhraseId,SentenceId,Phrase,Sentiment
0,1,1,A series of escapades demonstrating the adage ...,1
1,2,1,A series of escapades demonstrating the adage ...,2
2,3,1,A series,2
3,4,1,A,2
4,5,1,series,2
5,6,1,of escapades demonstrating the adage that what...,2
6,7,1,of,2
7,8,1,escapades demonstrating the adage that what is...,2
8,9,1,escapades,2
9,10,1,demonstrating the adage that what is good for ...,2


In [5]:
print(test.info())
test.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66292 entries, 0 to 66291
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   PhraseId    66292 non-null  int64 
 1   SentenceId  66292 non-null  int64 
 2   Phrase      66292 non-null  object
dtypes: int64(2), object(1)
memory usage: 1.5+ MB
None


,PhraseId,SentenceId,Phrase
0,156061,8545,An intermittently pleasing but mostly routine ...
1,156062,8545,An intermittently pleasing but mostly routine ...
2,156063,8545,An
3,156064,8545,intermittently pleasing but mostly routine effort
4,156065,8545,intermittently pleasing but mostly routine
5,156066,8545,intermittently pleasing but
6,156067,8545,intermittently pleasing
7,156068,8545,intermittently
8,156069,8545,pleasing
9,156070,8545,but


## **2.2 Columns**

- `PhraseId` : dataset序列. 鍵值.<br>
- `SentenceId` : 針對句子產生的序列編號.<br>
- `Phrase` : 文本資料.<br>
- `Sentiment` : 表示文本所帶有的情緒.只有traindata有這個列.<br>

在本次的項目當中<br>
將會預測testdata的句子(Phrase)所帶有的情緒.<br>

In [6]:
#顯示資料的columns

print(f'''
traindata columns :
{train.columns}
數量 : {len(train.columns)}

testdata columns :
{test.columns}
數量 : {len(test.columns)}
''')


traindata columns :
Index(['PhraseId', 'SentenceId', 'Phrase', 'Sentiment'], dtype='object')
數量 : 4

testdata columns :
Index(['PhraseId', 'SentenceId', 'Phrase'], dtype='object')
數量 : 3



### **2.2.1 PhraseId**

PhraseId為dataset的鍵值.


### **2.2.2 SentenceId**

SentenceId用於分類PharseId是屬於出自哪個序號的句子.<br>
經觀察後發現<br>
每個SentenceId除了包含最初始完整的句子<br>
同時也包含了同個SentenceId經過切割劃分的句子片段<br>
因此同時會有數筆資料屬於同個SentenceId.<br>


In [7]:

#顯示個別SentenceId和
#以及其原始未經過切割的文本
print('SentenceId and Pharase:')
for n in range(1,11):
    n=train[train['SentenceId']==n][['SentenceId','Phrase']].values[0][0]
    m=train[train['SentenceId']==n][['SentenceId','Phrase']].values[0][1]
    #print()
    print('-'*20)
    print(f'[{n}]',m)

#任意顯示某個SentenceId的資料
train[train['SentenceId']==3]
    

SentenceId and Pharase:
--------------------
[1] A series of escapades demonstrating the adage that what is good for the goose is also good for the gander , some of which occasionally amuses but none of which amounts to much of a story .
--------------------
[2] This quiet , introspective and entertaining independent is worth seeking .
--------------------
[3] Even fans of Ismail Merchant 's work , I suspect , would have a hard time sitting through this one .
--------------------
[4] A positively thrilling combination of ethnography and all the intrigue , betrayal , deceit and murder of a Shakespearean tragedy or a juicy soap opera .
--------------------
[5] Aggressive self-glorification and a manipulative whitewash .
--------------------
[6] A comedy-drama of nearly epic proportions rooted in a sincere performance by the title character undergoing midlife crisis .
--------------------
[7] Narratively , Trouble Every Day is a plodding mess .
--------------------
[8] The Importance of B

,PhraseId,SentenceId,Phrase,Sentiment
81,82,3,"Even fans of Ismail Merchant 's work , I suspe...",1
82,83,3,Even fans of Ismail Merchant 's work,2
83,84,3,Even fans,2
84,85,3,Even,2
85,86,3,fans,3
86,87,3,of Ismail Merchant 's work,2
87,88,3,Ismail Merchant 's work,2
88,89,3,Ismail Merchant 's,2
89,90,3,Ismail,2
90,91,3,Merchant 's,2


### **2.2.3 Phrase**

Phrase為文本<br>
包含了每個SentenceId的完整句子<br>
以及切割後的片段<br>

資料當中<br>
包含了非常多原始句子擷取的片段<br>
若是要理解到正確的語意以及其中帶有的情緒<br>
必須要考慮到完整的句<br>
而不是其中的個別的詞或片段.<br>
因此我們將以SentenceId為主<br>
篩選出完整的句子<br>
並觀察他們的sentiment數.<br>

此處必須更正想法<br>
這些被拆解的片段雖然不能用於表示原始句子的語意和情緒<br>
不過作為模型訓練的data仍然是有價值的<br>
或許這樣可以當作擴充訓練資料集的方法之一.<br>
因此<br>
更正之前的想法<br>
會將所有的資料都帶入模型當中做訓練.<br>

思考問題和想法:<br>
- 這邊我試著要抓出每個sentence完整的句子<br>
用的方法是從每個SentenceId抓出第一筆資料.<br>
那如果完整的句子不是第一筆呢?<br>
測試後發現<br>
有些SentenceId確實第一筆資料不是完整的句子<br>
必須再思考用甚麼方法挑出原始句子會更好.<br>

In [8]:
#挑出任意一個句子出來比對

print('SentenceId 16 原始句子:')
print(train[train['SentenceId']==16]['Phrase'].values[0])

print('')
train[train['SentenceId']==16]

SentenceId 16 原始句子:
A welcome relief from baseball movies that try too hard to be mythic , this one is a sweet and modest and ultimately winning story .



,PhraseId,SentenceId,Phrase,Sentiment
423,424,16,A welcome relief from baseball movies that try...,3
424,425,16,A welcome relief from baseball movies that try...,3
425,426,16,A welcome relief,3
426,427,16,welcome relief,3
427,428,16,welcome,2
428,429,16,relief,2
429,430,16,from baseball movies that try too hard to be m...,1
430,431,16,baseball movies that try too hard to be mythic,0
431,432,16,baseball movies,2
432,433,16,baseball,2


後面這段<br>
是之前嘗試挑出完整句子的操作<br>
不過現在用不到了.<br>

In [9]:
#測試
#測試用不同方法從train挑出來的資料會有甚麼差異


#找原始文本方法一.
#透過找出每個SentenceId最多字數的資料.

train1=pd.DataFrame(train,copy=True)
train1['word_counts']=train1['Phrase'].apply(len)
index=train1.groupby('SentenceId')['word_counts'].idxmax().tolist()

train1=train1.iloc[index]
train1=train1.reset_index(drop=True)
train1=train1[['PhraseId', 'SentenceId', 'Phrase', 'Sentiment']]

#train1

#----------------------------------
#找原始文本方法二.
#從每個SentenceId第一筆資料挑出.

sentence_list=[]

for n in train.groupby('SentenceId').count().index:
    n=train[train['SentenceId']==n].values[0].tolist()
    sentence_list.append(n)

train2=pd.DataFrame(sentence_list,columns=train.columns)
train2.rename(columns={"Phrase": "Sentence"})

#train2

,PhraseId,SentenceId,Sentence,Sentiment
0,1,1,A series of escapades demonstrating the adage ...,1
1,64,2,"This quiet , introspective and entertaining in...",4
2,82,3,"Even fans of Ismail Merchant 's work , I suspe...",1
3,117,4,A positively thrilling combination of ethnogra...,3
4,157,5,Aggressive self-glorification and a manipulati...,1
...,...,...,...,...
8524,155985,8540,... either you 're willing to go with this cla...,2
8525,155998,8541,"Despite these annoyances , the capable Claybur...",2
8526,156022,8542,-LRB- Tries -RRB- to parody a genre that 's al...,1
8527,156032,8543,The movie 's downfall is to substitute plot fo...,1


In [10]:
#測試

#找出最train['SentenceId']==16中字數最多的index
# x=train[train['SentenceId']==16]['Phrase'].apply(len)
# x.idxmax()

In [11]:
#測試

#下面函式用來檢查兩個DF是否相等
#如果不想等則會出錯.
#https://blog.csdn.net/qcyfred/article/details/102601971
# pd.testing.assert_frame_equal(train1,train2)
#執行後發現錯誤.

#分別檢查其中某個錯誤點是甚麼情況.
# train1[train1['PhraseId']==2007]
# train2[train1['PhraseId']==2006]
# train[train['SentenceId']==76]

#檢查後發現
#train2[train1['PhraseId']==2006]的Phrase的值為空值
#原因是train data中
#PhraseId為76的第一個序列資料就是這個空值
#而非我們預期的原始文本.

### **2.2.4 Sentiment**

Semtiment用於表示文本所帶有的情緒.<br>
test dataset沒有這個column<br>
而train dataset則有這個column.<br>
在本次的項目當中將會預測test dataset是屬於哪個Semtiment值<br>
Semtiment會作為模型的Y值.<br>

其值的含意:<br>
0 - negative<br>
1 - somewhat negative<br>
2 - neutral<br>
3 - somewhat positive<br>
4 - positive<br>

In [12]:
print('Sentiment值以及其數量:')
train['Sentiment'].value_counts().sort_index()

Sentiment值以及其數量:


0     7072
1    27273
2    79582
3    32927
4     9206
Name: Sentiment, dtype: int64

# **3. Data Preprocessing(資料前處理)**

將資料轉換成模型輸入的格式.<br>

資料沒有缺失值.<br>

思考問題和想法:<br>
- 也許可以嘗試看看不同的配置名稱.<br>
- 也許可以試試不同的語言模型<br>
並且了解他們的差異.<br>
- 如果語言模型遇到語料庫中沒有的詞會怎麼辦?<br>

In [13]:
#超參數

max_length=80
model_name='bert-base-uncased'

## **載入語言模型**

In [14]:
tokenizer=BertTokenizer.from_pretrained(model_name)
test_text=train.iloc[0]['Phrase']
clear_output()

#tokenizer其中10個tokens對應ids
print('-'*20)
for n in np.random.randint(0,len(tokenizer.get_vocab()),10):
    n=list(tokenizer.get_vocab().keys())[n] 
    print('{0:15}{1:1}'.format(n,tokenizer.get_vocab()[n]))

print('-'*20)
print('tokenizer轉換測試:')
print(test_text)
print(tokenizer.encode(test_text))
print(tokenizer.convert_ids_to_tokens(tokenizer.encode(test_text)))

--------------------
##olio         29401
1843           10075
adi            27133
rubble         17538
ufo            24321
##brate        22008
skater         18815
deformation    29130
handles        16024
janeiro        11497
--------------------
tokenizer轉換測試:
A series of escapades demonstrating the adage that what is good for the goose is also good for the gander , some of which occasionally amuses but none of which amounts to much of a story .
[101, 1037, 2186, 1997, 9686, 17695, 18673, 14313, 1996, 15262, 3351, 2008, 2054, 2003, 2204, 2005, 1996, 13020, 2003, 2036, 2204, 2005, 1996, 25957, 4063, 1010, 2070, 1997, 2029, 5681, 2572, 25581, 2021, 3904, 1997, 2029, 8310, 2000, 2172, 1997, 1037, 2466, 1012, 102]
['[CLS]', 'a', 'series', 'of', 'es', '##cap', '##ades', 'demonstrating', 'the', 'ada', '##ge', 'that', 'what', 'is', 'good', 'for', 'the', 'goose', 'is', 'also', 'good', 'for', 'the', 'gan', '##der', ',', 'some', 'of', 'which', 'occasionally', 'am', '##uses', 'but', 'none',

In [15]:
%%time
#觀察文本轉換後的各別token總數

tokens=[]
for n in range(len(train['Phrase'])):
    n=tokenizer.encode(train['Phrase'][n])
    tokens.append(n)   
#len(tokens)

#可能會需要視覺化句子token數量
#這樣可以比較知道padding數值要設為多少
#這邊可以用隨機取樣的方式
for n in tokens[:10]:
    print(len(n), n[:20],'...')
    
max_seq_len = max([len(n) for n in tokens])
print('句子最大tokens數: ',max_seq_len)
    

44 [101, 1037, 2186, 1997, 9686, 17695, 18673, 14313, 1996, 15262, 3351, 2008, 2054, 2003, 2204, 2005, 1996, 13020, 2003, 2036] ...
19 [101, 1037, 2186, 1997, 9686, 17695, 18673, 14313, 1996, 15262, 3351, 2008, 2054, 2003, 2204, 2005, 1996, 13020, 102] ...
4 [101, 1037, 2186, 102] ...
3 [101, 1037, 102] ...
3 [101, 2186, 102] ...
17 [101, 1997, 9686, 17695, 18673, 14313, 1996, 15262, 3351, 2008, 2054, 2003, 2204, 2005, 1996, 13020, 102] ...
3 [101, 1997, 102] ...
16 [101, 9686, 17695, 18673, 14313, 1996, 15262, 3351, 2008, 2054, 2003, 2204, 2005, 1996, 13020, 102] ...
5 [101, 9686, 17695, 18673, 102] ...
13 [101, 14313, 1996, 15262, 3351, 2008, 2054, 2003, 2204, 2005, 1996, 13020, 102] ...
句子最大tokens數:  80
CPU times: user 1min 6s, sys: 192 ms, total: 1min 6s
Wall time: 1min 6s


## **轉換文本資料**

In [16]:
#測試

#後面都還需要修改

In [17]:
%time
#轉換資料成model的input

#bert encoder.
#定義轉換資料的函式
#此函式將會把dataset轉換成model input
def encode(data):
    
    input_ids=[]
    attention_masks=[]

    for n in range(len(data)):
        #tokenizer.encode_plus 參數:
        #max_length 指定被補足的token數量
        #truncation 設定是否刪除超過max_length數量的token 
        input=tokenizer.encode_plus(data.Phrase[n],
                                    add_special_tokens=True,
                                    padding='max_length',
                                    max_length=max_length,
                                    truncation=True,
                                    return_attention_mask=True,)

        input_ids.append(input['input_ids'])
        attention_masks.append(input['attention_mask'])
    return np.array(input_ids), np.array(attention_masks)
        
#轉換資料        
input_ids, attention_masks=encode(train)
test_input_ids, test_attention_masks=encode(test)

#顯示處理後的資料
#model input
print('input_ids:')
for n in range(10):
    print(input_ids[n][:20],'...')
print('attention_masks:')
for n in range(10):
    print(attention_masks[n][:20],'...')

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.15 µs
input_ids:
[  101  1037  2186  1997  9686 17695 18673 14313  1996 15262  3351  2008
  2054  2003  2204  2005  1996 13020  2003  2036] ...
[  101  1037  2186  1997  9686 17695 18673 14313  1996 15262  3351  2008
  2054  2003  2204  2005  1996 13020   102     0] ...
[ 101 1037 2186  102    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0] ...
[ 101 1037  102    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0] ...
[ 101 2186  102    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0] ...
[  101  1997  9686 17695 18673 14313  1996 15262  3351  2008  2054  2003
  2204  2005  1996 13020   102     0     0     0] ...
[ 101 1997  102    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0] ...
[  101  9686 17695 18673 14313  1996 15262  3351  2008  2054  2003  2204
  2005  1996 13020   102     0     0     0 

# **4. Model**

改善想法:
- 最後可加入confusion_matrix<br>
- 嘗試用不同的預訓練模型<br>



思考問題和想法:
- 看這裡面有沒有可以用於改善model的想法<br>
概念篇 https://www.youtube.com/watch?v=xki61j7z-30&list=PLJV_el3uVTsPy9oCRY30oBPNLCo89yu49&index=16<br>
實作篇 https://www.youtube.com/watch?v=Ky1ku1miDow&list=PLJV_el3uVTsPy9oCRY30oBPNLCo89yu49&index=20<br>
- 我想把batch_sizes設定為1000做訓練<br>
不過出現錯誤了<br>
`ResourceExhaustedError : OOM when allocating ...`<br>
`failed to allocate memory`<br>
這是為甚麼?<br>
原因是記憶體不夠用了<br>
那其他更細節的原因呢?<br>
- 一次把所有資料帶進去訓練<br>
batech_sizes設定為100<br>
每個epochs時間要20分鐘.<br>
如果希望減少訓練時間而減少帶入模型的資料量<br>
怎樣的資料量是比較剛好的?<br>
資料量和訓練時間之間會呈現甚麼關係?<br>
- 模型會把資料分批帶入做訓練<br>
那在不同的epoch中<br>
這些分批的資料會是一樣的嗎?<br>
是否一樣會影響訓練的結果嗎?<br>
- batch_sizes不應該設太大<br>
因為會讓訓練的結果不好.<br>
其中的原因是甚麼?<br>
- 在這次的模型訓練當中<br>
第二輪以後val準確度便沒有再提升<br>
這個狀況在之前項目中也有出現.<br>
該怎麼理解這個狀況?<br>
如果某輪訓練沒有提升準確度<br>
那之後的訓練是否就不會再提升了?<br>
- model.compile(Adam(lr=6e-6), loss='binary_crossentropy', metrics=['accuracy'])<br>
如果想提升訓練結果<br>
這邊是否能做某些調整?<br>
那還應該做甚麼調整?<br>
我能透過dropout提高val data的表現嗎?
是否要提高BERT model中的dropout的比例才有用?
adam和loss這兩個參數我該怎麼理解?<br>
測是不同的optimization<br>
-這些是甚麼?<br>
local minimum<br>
saddle point<br>
plateau<br>
- 如果有A model和 B model<br>
A model有20個layers<br>
B model有50個layers<br>
那A model可以做到的事情(準確度夠高)<br>
B model自然也可以做到.<br>
那如果B model沒辦法有差不多的準確度<br>
這可能代表model沒有訓練好.<br>
- 如果模型的準確度不高<br>
可能會是甚麼問題?<br>
模型沒有訓練好.<br>
模型參數不夠多(架構不對、不夠複雜,這邊在說的都是同一個狀況).<br>
https://www.youtube.com/watch?v=xki61j7z-30&list=PLJV_el3uVTsPy9oCRY30oBPNLCo89yu49&index=16<br>
- 透過relu可以改善前面的隱藏層訓練不夠的問題<br>
- neruel的數量可以理解為input轉換後的數量嗎?<br>
- max pooling是甚麼?<br>
匯總某個範圍內的值<br>
並提取最大的值當作這個區域的代表值<br>
- maxout network這樣的概念用在模型裡面會有甚麼效果?<br>
maxout就是max pooling<br>
https://youtu.be/xki61j7z-30?t=1512<br>
- neruel輸出值的正負有甚麼涵義嗎?<br>
- 是否可以視覺化loss值<br>
以了解模型訓練的狀況(plateau saddle,point,local minima)<br>
- adam該怎麼定義他?<br>
有甚麼情況是會選擇不用adam的?<br>
- 我該如何知道訓練用的資料是否會超出記憶體?

In [18]:
#模型相關定義及超參數

x=[input_ids,attention_masks]
y=to_categorical(train[['Sentiment']],num_classes=5)
test_x=[test_input_ids, test_attention_masks]


## **BERT模型引入**

In [19]:
bertmodel1 = TFBertModel.from_pretrained('bert-base-uncased')

#顯示bertmodel配置
print('-'*20)
print(bertmodel1.config)

#顯示bertmodel 選定的隱藏層的配置
#bertmodel1.layers[n].get_config()
# bertmodel1.layers[0].get_config()

#顯示用bertmodel轉換後的值
# x1=tf.convert_to_tensor(input_ids)
# x2=tf.convert_to_tensor(attention_masks)
# bertmodel([x1,x2])

Downloading:   0%|          | 0.00/511M [00:00<?, ?B/s]

2022-07-07 09:17:50.934906: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-07-07 09:17:50.936172: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-07-07 09:17:50.936945: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-07-07 09:17:50.937949: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compil

--------------------
BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.18.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



## **模型建構**

### **model2**

In [20]:
#模型2

def model2(bertmodel):

    #模型架構
    
    ids = tf.keras.Input(shape=(80,),dtype='int64')
    masks = tf.keras.Input(shape=(80,),dtype='int64')
    input=[ids,masks]

    output=bertmodel([ids,masks])#輸出應該是word embedding. 
    output=output[1]
    output=tf.keras.layers.Dropout(0.5)(output)
    output=tf.keras.layers.Dense(5, activation='sigmoid')(output)   

    model=tf.keras.models.Model(inputs = input,outputs = output)
    model.compile(Adam(lr=6e-6), loss='binary_crossentropy', metrics=['accuracy'])
    return model
    

model2=model2(bertmodel1)
model2.summary()


Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 80)]         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 80)]         0                                            
__________________________________________________________________________________________________
tf_bert_model (TFBertModel)     TFBaseModelOutputWit 109482240   input_1[0][0]                    
                                                                 input_2[0][0]                    
__________________________________________________________________________________________________
dropout_37 (Dropout)            (None, 768)          0           tf_bert_model[0][1]          

/opt/conda/lib/python3.7/site-packages/keras/optimizer_v2/optimizer_v2.py:356: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  "The `lr` argument is deprecated, use `learning_rate` instead.")


In [21]:
#結果1

#batch_size=100
#資料數:約15萬

# Epoch 1/5
# 1249/1249 [==============================] - 1229s 984ms/step - loss: 0.2990 - accuracy: 0.6492 - val_loss: 0.2869 - val_accuracy: 0.6567
# Epoch 2/5
# 1249/1249 [==============================] - 1182s 946ms/step - loss: 0.2639 - accuracy: 0.6970 - val_loss: 0.2824 - val_accuracy: 0.6651
# Epoch 3/5
# 1249/1249 [==============================] - 1183s 947ms/step - loss: 0.2481 - accuracy: 0.7195 - val_loss: 0.2838 - val_accuracy: 0.6621
# Epoch 4/5
# 1249/1249 [==============================] - 1183s 947ms/step - loss: 0.2367 - accuracy: 0.7377 - val_loss: 0.2887 - val_accuracy: 0.6606
# Epoch 5/5
# 1249/1249 [==============================] - 1184s 948ms/step - loss: 0.2271 - accuracy: 0.7508 - val_loss: 0.2915 - val_accuracy: 0.6592

### **model1**

In [22]:
#模型1

def model1(bertmodel):

    #模型架構
    
    ids = tf.keras.Input(shape=(80,),dtype='int32')
    masks = tf.keras.Input(shape=(80,),dtype='int32')
    input=[ids,masks]

    output=bertmodel([ids,masks])#輸出應該是word embedding. 
    output=output[1]
    output=tf.keras.layers.Dropout(0.75)(output)
    output=tf.keras.layers.Dense(5, activation='sigmoid')(output)   

    model=tf.keras.models.Model(inputs = input,outputs = output)
    model.compile(Adam(lr=6e-6), loss='CategoricalCrossentropy', metrics=['accuracy'])
    return model
    

model1=model1(bertmodel1)
model1.summary()



Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 80)]         0                                            
__________________________________________________________________________________________________
input_4 (InputLayer)            [(None, 80)]         0                                            
__________________________________________________________________________________________________
tf_bert_model (TFBertModel)     TFBaseModelOutputWit 109482240   input_3[0][0]                    
                                                                 input_4[0][0]                    
__________________________________________________________________________________________________
dropout_38 (Dropout)            (None, 768)          0           tf_bert_model[1][1]        

In [23]:
#結果1

#batch_size=100
#資料數:15606
#loss='binary_crossentropy'
#Dropout=0.5

#想法:
#如果把資料變多
#有可能改善準確度嗎?

# Epoch 1/4
# 125/125 [==============================] - 121s 971ms/step - loss: 0.4634 - accuracy: 0.4894 - val_loss: 0.3986 - val_accuracy: 0.5577
# Epoch 2/4
# 125/125 [==============================] - 119s 949ms/step - loss: 0.4083 - accuracy: 0.5561 - val_loss: 0.3979 - val_accuracy: 0.5577
# Epoch 3/4
# 125/125 [==============================] - 118s 948ms/step - loss: 0.4032 - accuracy: 0.5577 - val_loss: 0.3977 - val_accuracy: 0.5577
# Epoch 4/4
# 125/125 [==============================] - 118s 948ms/step - loss: 0.4019 - accuracy: 0.5572 - val_loss: 0.3984 - val_accuracy: 0.5577

In [24]:
#結果2

#batch_size=100
#資料數:15606
#loss='CategoricalCrossentropy'
#Dropout=0.5

# Epoch 1/20
# 125/125 [==============================] - 122s 973ms/step - loss: 1.2144 - accuracy: 0.5578 - val_loss: 1.2162 - val_accuracy: 0.5577
# Epoch 2/20
# 125/125 [==============================] - 122s 976ms/step - loss: 1.2107 - accuracy: 0.5578 - val_loss: 1.2170 - val_accuracy: 0.5577
# Epoch 3/20
# 125/125 [==============================] - 122s 973ms/step - loss: 1.2028 - accuracy: 0.5574 - val_loss: 1.2251 - val_accuracy: 0.5577
# Epoch 4/20
# 125/125 [==============================] - 121s 970ms/step - loss: 1.1924 - accuracy: 0.5575 - val_loss: 1.2334 - val_accuracy: 0.5570
# Epoch 5/20
# 125/125 [==============================] - 120s 964ms/step - loss: 1.1727 - accuracy: 0.5585 - val_loss: 1.2563 - val_accuracy: 0.5525
# Epoch 6/20
# 125/125 [==============================] - 121s 969ms/step - loss: 1.1396 - accuracy: 0.5702 - val_loss: 1.3018 - val_accuracy: 0.5288

In [25]:
#結果3

#batch_size=100
#資料數:15606
#loss='CategoricalCrossentropy'
#Dropout=0.2

# Epoch 1/20
# 125/125 [==============================] - 140s 983ms/step - loss: 1.3519 - accuracy: 0.4928 - val_loss: 1.2273 - val_accuracy: 0.5577
# Epoch 2/20
# 125/125 [==============================] - 119s 955ms/step - loss: 1.2246 - accuracy: 0.5563 - val_loss: 1.2250 - val_accuracy: 0.5577
# Epoch 3/20
# 125/125 [==============================] - 119s 954ms/step - loss: 1.2189 - accuracy: 0.5576 - val_loss: 1.2179 - val_accuracy: 0.5577
# Epoch 4/20
# 125/125 [==============================] - 119s 955ms/step - loss: 1.2126 - accuracy: 0.5577 - val_loss: 1.2254 - val_accuracy: 0.5577
# Epoch 5/20
# 125/125 [==============================] - 119s 955ms/step - loss: 1.2072 - accuracy: 0.5581 - val_loss: 1.2306 - val_accuracy: 0.5577
# Epoch 6/20
# 125/125 [==============================] - 119s 955ms/step - loss: 1.1974 - accuracy: 0.5575 - val_loss: 1.2345 - val_accuracy: 0.5577
# Epoch 7/20
# 125/125 [==============================] - 119s 956ms/step - loss: 1.1804 - accuracy: 0.5582 - val_loss: 1.2359 - val_accuracy: 0.5570
# Epoch 8/20
# 125/125 [==============================] - 119s 955ms/step - loss: 1.1615 - accuracy: 0.5614 - val_loss: 1.2529 - val_accuracy: 0.5452



In [26]:
#結果4

#batch_size=100
#資料數:15606
#loss='CategoricalCrossentropy'
#Dropout=0.75

# Epoch 1/20
# 125/125 [==============================] - 142s 983ms/step - loss: 1.3812 - accuracy: 0.4856 - val_loss: 1.2492 - val_accuracy: 0.5580
# Epoch 2/20
# 125/125 [==============================] - 119s 955ms/step - loss: 1.1805 - accuracy: 0.5573 - val_loss: 1.2972 - val_accuracy: 0.5253
# Epoch 3/20
# 125/125 [==============================] - 119s 954ms/step - loss: 1.0909 - accuracy: 0.5881 - val_loss: 1.4019 - val_accuracy: 0.4862

### **model3**

In [27]:
#模型1

def model3(bertmodel):

    #模型架構
    
    ids = tf.keras.Input(shape=(80,),dtype='int64')
    masks = tf.keras.Input(shape=(80,),dtype='int64')
    input=[ids,masks]

    output=bertmodel([ids,masks])#輸出應該是word embedding. 
    output=output[1]
    output=tf.keras.layers.Dropout(0.5)(output)
    output=tf.keras.layers.Dense(768, activation='relu')(output) 
    output=tf.keras.layers.Dropout(0.5)(output)
    output=tf.keras.layers.Dense(768, activation='relu')(output) 
    output=tf.keras.layers.Dropout(0.5)(output)
    output=tf.keras.layers.Dense(768, activation='relu')(output) 
    output=tf.keras.layers.Dropout(0.5)(output)
    output=tf.keras.layers.Dense(5, activation='sigmoid')(output)   

    model=tf.keras.models.Model(inputs = input,outputs = output)
    model.compile(Adam(lr=6e-6), loss='CategoricalCrossentropy', metrics=['accuracy'])
    return model
    

model3=model3(bertmodel1)
model3.summary()

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_5 (InputLayer)            [(None, 80)]         0                                            
__________________________________________________________________________________________________
input_6 (InputLayer)            [(None, 80)]         0                                            
__________________________________________________________________________________________________
tf_bert_model (TFBertModel)     TFBaseModelOutputWit 109482240   input_5[0][0]                    
                                                                 input_6[0][0]                    
__________________________________________________________________________________________________
dropout_39 (Dropout)            (None, 768)          0           tf_bert_model[2][1]        

In [28]:
# Epoch 1/20
# 125/125 [==============================] - 136s 971ms/step - loss: 1.3418 - accuracy: 0.4581 - val_loss: 1.3922 - val_accuracy: 0.5086
# Epoch 2/20
# 125/125 [==============================] - 118s 946ms/step - loss: 1.0994 - accuracy: 0.5724 - val_loss: 1.4869 - val_accuracy: 0.4363
# Epoch 3/20
# 125/125 [==============================] - 118s 946ms/step - loss: 1.0377 - accuracy: 0.6028 - val_loss: 1.5102 - val_accuracy: 0.4513
# Epoch 4/20
# 125/125 [==============================] - 118s 946ms/step - loss: 0.9853 - accuracy: 0.6208 - val_loss: 1.6007 - val_accuracy: 0.4218
# Epoch 5/20
# 125/125 [==============================] - 118s 945ms/step - loss: 0.9469 - accuracy: 0.6343 - val_loss: 1.6861 - val_accuracy: 0.4071
# Epoch 6/20
#  59/125 [=============>................] - ETA: 57s - loss: 0.7705 - accuracy: 0.7161

## **訓練**

In [29]:
#抓取指定數量的隨機資料

# sample=np.random.randint(0,len(x[0]),size=round(0.1*len(x[0])))
# x=[x[0][sample],x[1][sample]]

# x[0].shape

In [30]:
%%time
#訓練模型

history2 = model2.fit(
    x,
    y,
    validation_split=0.2,
    epochs=3,
    batch_size=100)


2022-07-07 09:18:12.077836: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


Epoch 1/3
1249/1249 [==============================] - 1202s 949ms/step - loss: 0.3230 - accuracy: 0.6224 - val_loss: 0.2926 - val_accuracy: 0.6497
Epoch 2/3
1249/1249 [==============================] - 1184s 948ms/step - loss: 0.2708 - accuracy: 0.6896 - val_loss: 0.2845 - val_accuracy: 0.6605
Epoch 3/3
1249/1249 [==============================] - 1181s 946ms/step - loss: 0.2546 - accuracy: 0.7135 - val_loss: 0.2853 - val_accuracy: 0.6635
CPU times: user 38min 41s, sys: 4min 27s, total: 43min 9s
Wall time: 59min 38s


In [31]:
print('-'*30)
print('model2')
for n in history2.history:
    print('{}: {}'.format(n,history2.history[n]))

------------------------------
model2
loss: [0.32304316759109497, 0.270793616771698, 0.2546004056930542]
accuracy: [0.6223567724227905, 0.6896466016769409, 0.7135396599769592]
val_loss: [0.2925509810447693, 0.2844845652580261, 0.2853403091430664]
val_accuracy: [0.649653971195221, 0.6605151891708374, 0.6634948253631592]


## **提交test dataset預測結果**

In [32]:
#預測test dataset結果

test_y=model2.predict(test_x)
print('test_y')
print(test_y.argmax(1))

test_y
[3 3 2 ... 2 2 1]


In [33]:
#預測的結果合併到test中
test['Sentiment']=test_y.argmax(1)

#挑選幾筆資料來查看.
#這邊可以修改為隨機取樣的方式來查看.
num=0
start=1000
for n,m in zip(test['Sentiment'].iloc[start:],test['Phrase'].iloc[start:]):
    num+=1
    print(f'[{n}]',m)
    if num==20:
        break

[2] defoliation
[2] of ego
[2] ego
[3] make the film touching despite some doldrums
[3] make the film touching
[2] make
[3] the film touching
[3] film touching
[3] touching
[2] despite some doldrums
[2] some doldrums
[2] some
[2] doldrums
[0] Imagine a really bad community theater production of West Side Story without the songs .
[0] Imagine a really bad community theater production of West Side Story without the songs
[0] Imagine a really bad community theater production of West Side Story
[2] Imagine
[0] a really bad community theater production of West Side Story
[0] a really bad community theater production
[0] really bad community theater production


In [34]:
#將預測結果合併至競賽提交格式

submission=pd.read_csv('../input/sentiment-analysis-on-movie-reviews/sampleSubmission.csv')

submission['Sentiment']=test['Sentiment']
submission.to_csv('submission.csv',index=False)

#最終競賽提交格式
submission.head(10)

,PhraseId,Sentiment
0,156061,3
1,156062,3
2,156063,2
3,156064,3
4,156065,3
5,156066,3
6,156067,3
7,156068,2
8,156069,3
9,156070,2
